===================================
 # PEAKS DETECTION #
===================================

Read the chromatogram, generate a peak table and identify the compounds using the NIST database. 

In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser
from pathlib import Path
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess
import sys
import threading


class GCGCMSAnalysisUI(Interface):
    """
     GC×GC-MS Analysis UI with improved error handling and flexible file/folder selection.
    Provides a widget-based interface for configuring and running GCGCMS analysis.
    Users can select individual files, folders, or subfolders - all compatible files will be processed.
    """
    
    def __init__(self):
        """Initialize the GCMS Analysis UI with default parameters and widgets."""
        super().__init__(supported_extensions=('.h5', '.cdf')) # respecter la logique  de priorite, le 1er de la liste = celui qui sera privilegie lors du choix entre les 2 extentions
        self._setup_default_parameters()
        self._create_parameter_widgets()
        self._create_gcgcms_widgets()
        self._create_base_widgets()
        self._create_action_widgets()
        self._setup_callbacks()
        self._setup_environment()

        
    def _setup_default_parameters(self):
        """Initialize default analysis parameters."""
        # Public parameters (configurable via UI)
        self.abs_threshold = "0"
        self.rel_threshold = "0.005"
        self.noise_factor = "1.5"
        self.min_persistence = "0.0002"
        
        # Private parameters (fixed for this UI)
        self._min_distance = 1
        self._sigma_ratio = 1.6
        self._num_sigma = 10
        self._min_sigma = 1
        self._max_sigma = 30
        self._overlap = 0.5
        self._match_factor_min = 650
        self._cluster = True
        self._min_samples = 4
        self._eps = 3
        self.formated_spectra = True #TODO ?
        
    
    def _create_parameter_widgets(self):
        """Create parameter input widgets."""
        self.w_noise_factor = widgets.Text(value=self.noise_factor)
        self.noise_factor = self._bold_widget("Noise factor", self.w_noise_factor)
        self.noise_factor_def = self.create_help_text(
            "Noise scaling factor used to filter detected peaks."
            "A peak is retained if its intensity is greater than the maximum intensity multiplied by this factor."
        )
        
        self.w_min_persistence = widgets.Text(value=self.min_persistence)
        self.min_persistence = self._bold_widget("Minimum persistence", self.w_min_persistence)
        self.min_persistence_def = self.create_help_text(
            "Minimum topological persistence threshold that a peak must exceed to be considered a true signal rather than noise."
        )
        
        self.w_abs_threshold = widgets.Text(value=self.abs_threshold)
        self.abs_threshold = self._bold_widget("Absolute threshold", self.w_abs_threshold)
        self.abs_threshold_def = self.create_help_text(
            "Absolute threshold used to filter detected peaks based on their raw intensity."
        )
        
        self.w_rel_threshold = widgets.Text(value=self.rel_threshold)
        self.rel_threshold = self._bold_widget("Relative threshold", self.w_rel_threshold)
        self.rel_threshold_def = self.create_help_text(
            "Relative threshold used to filter detected peaks based on their relative intensity."
        )
        
    def create_method_widgets(self):
        """Create method selection widgets."""
        label_method = widgets.HTML(value="<b>Peak Detection Method</b>")
        method_radio = widgets.RadioButtons(
            options=['persistent_homology', 'peak_local_max', 'LoG', 'DoG', 'DoH'],
            value='persistent_homology',
            description='',
            disabled=False
        )
        method_widget = widgets.VBox([label_method, method_radio])
        
        label_mode = widgets.HTML(value="<b>Analysis Mode</b>")
        mode_radio = widgets.RadioButtons(
            options=['tic', 'mass_per_mass', '3D'],
            value='tic',
            description='',
            disabled=False
        )
        mode_widget = widgets.VBox([label_mode, mode_radio])
    
        return method_widget, mode_widget, method_radio, mode_radio 


    def _create_gcgcms_widgets(self):
        """Create GC×GC-MS specific widgets."""
        # Title
        self.txt_title = widgets.HTML('<H1>GC×GC-MS Analysis Configuration</H1>')
        
        # Method and mode widgets
        self.w_method, self.w_mode, self.r_method, self.r_mode = self.create_method_widgets()
        
       # NIST matching
        self.nist = widgets.Checkbox(
            value=True,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        ) 
        
        # Action widgets
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()

    def _validate_parameters(self):
        """Validate input parameters."""
        errors = []
        
        try:
            noise_val = float(self.w_noise_factor.value)
            if noise_val < 0:
                errors.append("The noise factor must be non-negative")
        except ValueError:
            errors.append("The noise factor must be a valid number")
        
        try:
            pers_val = float(self.w_min_persistence.value)
            if pers_val < 0:
                errors.append("Minimum persistence must be non-negative")
        except ValueError:
            errors.append("Minimum persistence must be a valid number")
        
        try:
            abs_val = float(self.w_abs_threshold.value)
            if abs_val < 0:
                errors.append("Absolute threshold must be non-negative")
        except ValueError:
            errors.append("Absolute threshold must be a valid number")
        
        try:
            rel_val = float(self.w_rel_threshold.value)
            if rel_val < 0 or rel_val > 1:
                errors.append("Relative threshold must be between 0 and 1")
        except ValueError:
            errors.append("Relative threshold must be a valid number")
        
        return errors

    
    def get_all_files_from_selections(self):
        """
        Retrieves all supported files from all selections.
        Automatically determines whether it's a file or a folder.
        """
        all_files = []
        processed_paths = set()
        already_seen_files = set()

        for i, fc in enumerate(self._choosers):
            selected = fc.selected_path
            if not selected:
                continue

            try:
                selected_path = Path(selected)

                if str(selected_path) in processed_paths:
                    continue
                processed_paths.add(str(selected_path))

                if fc.selected_filename:
                    name_without_ext = fc.selected_filename.rsplit('.', 1)[0]

                    if fc.selected_filename.endswith(".cdf") and name_without_ext in already_seen_files:
                        print(f"⚠️  File already processed: {name_without_ext}.cdf")
                        continue    

                    if fc.selected_filename.endswith(".h5"):
                        already_seen_files.add(name_without_ext)

                    if fc.selected_filename.endswith(self.supported_extensions):
                        full_path = selected_path / fc.selected_filename
                        all_files.append(str(full_path))
                        # print(f"📄 File added: {full_path}")
                    else:
                        print(f"⚠️  Unsupported file ignored: {fc.selected_filename}")
                        print(f"   Supported extensions: {', '.join(self.supported_extensions)}")

                else: # ce n 'est pas un fichier
                    # print(f"📁 Processing folder: {selected_path}")
                    dir_files = self._get_files_from_directory(selected_path)

                    for f in dir_files:
                        path = str(Path(f))
                        if path not in processed_paths:
                            all_files.append(path)
                            processed_paths.add(path)
                    # print(f"   Found {len(dir_files)} compatible files")

            except Exception as e:
                print(f"❌ Error while processing selection '{selected}': {e}")

        return all_files

    
    def _on_button_click(self, b):
        """Handle button click event to start analysis."""
        with self.output:
            self.output.clear_output()
            print("🚀 Initializing GC×GC-MS analysis...")
            
            # Valider les paramètres
            errors = self._validate_parameters()
            if errors:
                print("❌ Parameter validation failed:")
                for error in errors:
                    print(f"  • {error}")
                return
            
            if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
                print("Output directory cannot be empty")
                return
            
            # Obtenir tous les fichiers sélectionnés
            print("\n📂 Collecting files from selections...")
            selected_files = self.get_all_files_from_selections()
            
            if not selected_files:
                print("❌ No compatible files (.cdf or .h5) found in selections.")
                print("💡 Please select files or folders containing .cdf or .h5 files.")
                return
            
            print(f"\n✅ {len(selected_files)} compatible files found")

            self._start_subprocess_analysis(selected_files)
            
    
    def _start_subprocess_analysis(self, selected_files):
        """Start the analysis in a separate subprocess to keep the UI responsive."""
        analysis_params = [
                sys.executable,
                "/app/src/peak_detection_analyze_cli.py", #ds le docker
                "--output", self.get_output_path(),
                "--method", self.r_method.value,
                "--mode", self.r_mode.value,
                "--noise_factor", (self.w_noise_factor.value),
                "--min_persistence", (self.w_min_persistence.value),
                "--abs_threshold", (self.w_abs_threshold.value),
                "--rel_threshold", (self.w_rel_threshold.value),
                "--min_distance", self._min_distance,
                "--min_sigma", self._min_sigma,
                "--max_sigma", self._max_sigma,
                "--sigma_ratio", self._sigma_ratio,
                "--num_sigma", self._num_sigma,
                "--match_factor_min", self._match_factor_min,
                "--overlap", self._overlap,
                "--eps", self._eps,
                "--min_samples", self._min_samples,
            ]
        #cas des booleens
        if self._cluster:
            analysis_params.append("--cluster")
        if self.formated_spectra:
            analysis_params.append("--formated_spectra")
        if self.nist.value:
            analysis_params.append("--nist")
        # cas des listes
        analysis_params += ["--input"] + selected_files

        analysis_params = list(map(str, analysis_params))

        # print("▶️ Lancement en tâche de fond :", " ".join(analysis_params))

        self.current_process = subprocess.Popen(
            analysis_params,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        self.stop_button.disabled = False 

        def stream_output(proc, output_widget):
            try:
                for line in iter(proc.stdout.readline, ''):
                    with output_widget:
                        print(line, end='')
                    # Vérifie si le process est terminé après chaque ligne
                    if proc.poll() is not None:
                        break
            except Exception as e:
                with output_widget:
                    print(f"⚠️ Error reading process output: {e}")
            finally:
                proc.stdout.close()
                retcode = proc.wait()
                with output_widget:
                    if retcode == 0:
                        print("\n✅ Analyse terminée avec succès")
                    else:
                        print(f"\n❌ L'analyse a échoué avec le code de retour {retcode}")
                        print(f"\n{'='*60}")
                self.stop_button.disabled = True
                self.current_process = None

        # thread pour afficher stdout en direct
        threading.Thread(
            target=stream_output,
            args=(self.current_process,
                  self.output),
                  daemon=True
            ).start()
        

    def display(self):
        """Display the complete UI."""
        display(
            self.txt_title,
            widgets.VBox([self._vbox, self._vbox2]),
            widgets.HBox([self.w_method, self.w_mode]),
            self.nist,
            self.noise_factor,
            self.noise_factor_def,
            self.min_persistence,
            self.min_persistence_def,
            self.abs_threshold,
            self.abs_threshold_def,
            self.rel_threshold,
            self.rel_threshold_def,
            widgets.HBox([self.run_button, self.stop_button, self.clear_button]),
            self.output
        )

In [ ]:
gcms_ui = GCGCMSAnalysisUI()
gcms_ui.display()